# 2차 모델 (v2-lite) - CatBoost

핵심 아이디어:
- 1차 모델(구조적/선택형) + "안전한" 추가 정보만 붙여서 성능 개선
- 다중선택 컬럼은 조합 폭발을 막기 위해 원문을 쓰지 않고 '선택 개수'로 요약
- 고유값 많은 school1/major_field는 Top-N + OTHER로 압축 후 사용
- 자유서술(텍스트) 컬럼은 v2-lite에서 제외 (v2-final에서 TF-IDF 등으로 별도 처리)

출력:
- Valid 성능(F1/Precision/Recall)
- threshold 탐색 결과표


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

from catboost import CatBoostClassifier


In [ ]:
# ✅ 본인 경로로 수정
train_path = "train.csv"   # <-- 수정
train = pd.read_csv(train_path)

TARGET = "completed"

print("train shape:", train.shape)
display(train.head())


In [ ]:
"""
[1차에서 완전 제거/제외했던 것들]
- ID, contest_award, idea_contest, generation, class3, class4: 1차 제외 유지
- 자유서술: interested_company, incumbents_lecture_scale_reason => v2-lite에서도 제외
- 다중선택 원문: desired_job, expected_domain, desired_certificate => v2-lite에서는 '선택 개수'로 대체
- onedayclass_topic, certificate_acquisition: v2-lite에서는 원문 제외(필요 시 개수 추가 가능)
- school1, major_field: v2-lite에서 압축 후 사용 (Top-N + OTHER)
"""

# 1차에서 완전 제거/제외 (v2-lite에서도 원문은 그대로 제외)
drop_remove = [
    "ID", "contest_award", "idea_contest", "generation",
    "class3", "class4",
]

drop_exclude_text = [
    "interested_company",
    "incumbents_lecture_scale_reason",
]

# 다중선택 원문은 v2-lite에서 직접 사용하지 않음 (대신 파생변수 사용)
multi_select_cols = ["desired_job", "expected_domain", "desired_certificate"]

# 기타 다중선택/폭발 가능 컬럼 (원문 제외)
other_multi_cols = ["onedayclass_topic", "certificate_acquisition"]

# v2-lite에서 "압축 후" 사용할 고유값 큰 컬럼
high_card_cols = ["school1", "major_field"]

# 실제 존재하는 컬럼만 남김
drop_remove = [c for c in drop_remove if c in train.columns]
drop_exclude_text = [c for c in drop_exclude_text if c in train.columns]
multi_select_cols = [c for c in multi_select_cols if c in train.columns]
other_multi_cols = [c for c in other_multi_cols if c in train.columns]
high_card_cols = [c for c in high_card_cols if c in train.columns]

# v2-lite에서 원문으로 제외할 컬럼들
DROP_COLS_V2 = list(set(drop_remove + drop_exclude_text + multi_select_cols + other_multi_cols))

print("DROP_COLS_V2 count:", len(DROP_COLS_V2))
print("multi_select_cols:", multi_select_cols)
print("high_card_cols:", high_card_cols)


In [ ]:
def count_multiselect(x, sep=","):
    """
    다중선택 문자열을 '선택 개수'로 변환.
    - NaN이면 0
    - 문자열이면 sep로 split 후 유효 토큰 개수
    - sep가 다른 경우(예: '|', ';')면 아래 sep를 바꿔야 함
    """
    if pd.isna(x):
        return 0
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return 0

    parts = [p.strip() for p in s.split(sep)]
    parts = [p for p in parts if p != ""]
    return len(parts)


In [ ]:
def topn_other_series(s: pd.Series, topn=20, other_label="OTHER"):
    """
    빈도 상위 topn만 유지하고 나머지는 OTHER로 묶는 함수.
    - train 기준으로 top set을 만들고 valid/test에도 동일 적용해야 함.
    """
    vc = s.value_counts(dropna=False)
    top = vc.head(topn).index
    return s.where(s.isin(top), other_label), set(top)


In [ ]:
# ✅ 기본 X, y 구성: v2-lite에서 제외할 원문 컬럼 제거
X = train.drop(columns=[c for c in ([TARGET] + DROP_COLS_V2) if c in train.columns]).copy()
y = train[TARGET].copy()

# ✅ class1, class2 포함 확인(요구사항)
assert "class1" in X.columns, "class1이 X에 없습니다. DROP_COLS_V2/원본 컬럼명 확인 필요"
assert "class2" in X.columns, "class2가 X에 없습니다. DROP_COLS_V2/원본 컬럼명 확인 필요"

print("Base X shape (v2-lite, before adding features):", X.shape)

# ✅ (A) 다중선택 파생변수 추가: 선택 개수
# - 원문은 제외했지만, 요약한 수치형 feature로 추가
# - 구분자 sep는 데이터 형태에 맞게 수정 가능 (기본 ',')
for col in multi_select_cols:
    new_col = f"n_{col}"
    X[new_col] = train[col].apply(lambda v: count_multiselect(v, sep=","))  # sep 필요 시 변경

# ✅ (B) 기타 다중선택(있다면)도 선택 개수만 추가(선택)
# - v2-lite에서는 필수는 아니지만, 효과 있을 수 있음
for col in other_multi_cols:
    new_col = f"n_{col}"
    X[new_col] = train[col].apply(lambda v: count_multiselect(v, sep=","))

print("X shape after adding multi-select counts:", X.shape)

# ✅ (C) 고유값 큰 컬럼(학교/세부전공)을 Top-N + OTHER로 압축해 추가
# - v2에서는 이 정보를 쓰되, 과적합 방지를 위해 압축
TOPN = 20
top_sets = {}  # valid/test 동일 적용을 위해 top set 저장

for col in high_card_cols:
    # train 전체 기준으로 topn set 만들고, X에 반영
    compressed, top = topn_other_series(train[col].astype("string").fillna("__MISSING__"), topn=TOPN)
    X[col] = compressed
    top_sets[col] = top

print("X shape after adding compressed high-card columns:", X.shape)


In [ ]:
# ✅ 결측률 컷으로 지나치게 결측 많은 컬럼 제거 (안정화)
MISSING_CUT = 0.6
missing_rate = X.isnull().mean().sort_values(ascending=False)
drop_miss_cols = missing_rate[missing_rate > MISSING_CUT].index.tolist()

print("결측 컷으로 제거될 컬럼 수:", len(drop_miss_cols))
if drop_miss_cols:
    print("drop_miss_cols:", drop_miss_cols)

X = X.drop(columns=drop_miss_cols)
print("X shape after missing cut:", X.shape)

# ✅ split (stratify)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ✅ categorical 컬럼 지정
cat_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# ✅ 범주형 결측은 "__MISSING__"로 통일
for df in (X_train, X_valid):
    if len(cat_features) > 0:
        df[cat_features] = df[cat_features].astype("string").fillna("__MISSING__")

print("cat_features count:", len(cat_features))
print("cat_features sample:", cat_features[:10])

print("y_train ratio:\n", y_train.value_counts(normalize=True))
print("y_valid ratio:\n", y_valid.value_counts(normalize=True))


In [ ]:
# ✅ v2-lite 모델: 기본은 v1과 동일하게 가져가되, 피처가 늘었으니 depth는 6 유지 권장
model_v2 = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=200
)

# 1) OTHER 같은 문자열이 들어간 컬럼을 string으로 강제
force_cat_cols = []

# v2에서 OTHER를 만든 압축 컬럼들 (있을 때만)
for c in ["school1", "major_field"]:
    if c in X_train.columns:
        force_cat_cols.append(c)

# 혹시 다른 컬럼에도 OTHER를 넣었다면 여기에 추가
# force_cat_cols += ["원하는컬럼명"]

for df in [X_train, X_valid]:
    for c in force_cat_cols:
        df[c] = df[c].astype("string").fillna("__MISSING__")

# 2) cat_features에 강제로 포함 (중요)
cat_features = X_train.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()

# 3) 안전 점검: 수치형 컬럼에 문자열이 끼어있는지 체크
num_cols = X_train.select_dtypes(include=[np.number]).columns
bad_num = []
for c in num_cols:
    # 숫자형인데 문자열이 섞인 경우가 있으면 dtype이 object로 바뀌는 게 일반적이지만,
    # 혹시 캐스팅 문제로 남아있으면 여기서 잡힘
    if X_train[c].astype("string").str.contains("OTHER", na=False).any():
        bad_num.append(c)

print("force_cat_cols:", force_cat_cols)
print("cat_features count:", len(cat_features))
print("numeric cols containing 'OTHER':", bad_num)


model_v2.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

print("best_iteration:", model_v2.get_best_iteration())


In [ ]:
proba_valid = model_v2.predict_proba(X_valid)[:, 1]

def metrics_at_thr(t):
    pred = (proba_valid >= t).astype(int)
    return {
        "thr": float(t),
        "f1": f1_score(y_valid, pred, zero_division=0),
        "precision": precision_score(y_valid, pred, zero_division=0),
        "recall": recall_score(y_valid, pred, zero_division=0),
        "positive_rate": float(pred.mean()),
    }

grid = np.arange(0.05, 0.96, 0.01)
thr_df = pd.DataFrame([metrics_at_thr(t) for t in grid]).sort_values("f1", ascending=False)

display(thr_df.head(20))

best_thr = float(thr_df.iloc[0]["thr"])
best_f1 = float(thr_df.iloc[0]["f1"])
print("Best thr:", best_thr)
print("Best F1 :", round(best_f1, 4))


In [ ]:
FINAL_THR = 0.47
  # 사용자 합의: 운영용 threshold 고정

pred_valid_final = (proba_valid >= FINAL_THR).astype(int)

print("=== Valid Report @ FINAL_THR (v2-lite) ===")
print("F1       :", round(f1_score(y_valid, pred_valid_final, zero_division=0), 4))
print("Precision:", round(precision_score(y_valid, pred_valid_final, zero_division=0), 4))
print("Recall   :", round(recall_score(y_valid, pred_valid_final, zero_division=0), 4))
print()
print(classification_report(y_valid, pred_valid_final, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_valid, pred_valid_final))


In [ ]:
import pandas as pd
import numpy as np

test = pd.read_csv("test.csv")                 # 경로 수정
sub  = pd.read_csv("sample_submission.csv")    # 경로 수정

assert "ID" in test.columns, "test에 ID가 없습니다."
assert "ID" in sub.columns, "sample_submission에 ID가 없습니다."
print("test:", test.shape, "sub:", sub.shape)


In [ ]:
# --- (1) 원문 제외 컬럼 제거 ---
X_test = test.drop(columns=[c for c in DROP_COLS_V2 if c in test.columns], errors="ignore").copy()

# --- (2) 다중선택 -> 선택 개수 파생 ---
def count_multiselect(x, sep=","):
    if pd.isna(x):
        return 0
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return 0
    parts = [p.strip() for p in s.split(sep)]
    parts = [p for p in parts if p != ""]
    return len(parts)

for col in multi_select_cols:
    X_test[f"n_{col}"] = test[col].apply(lambda v: count_multiselect(v, sep=",")) if col in test.columns else 0

for col in other_multi_cols:
    X_test[f"n_{col}"] = test[col].apply(lambda v: count_multiselect(v, sep=",")) if col in test.columns else 0

# --- (3) high-card -> Top-N + OTHER (train에서 만든 top_sets 기준) ---
for col in high_card_cols:
    if col in test.columns:
        s = test[col].astype("string").fillna("__MISSING__")
        top = top_sets.get(col, set())
        X_test[col] = s.where(s.isin(top), "OTHER").astype("string")
    else:
        X_test[col] = "OTHER"

# --- (4) 결측컷으로 제거했던 컬럼 제거 ---
X_test = X_test.drop(columns=[c for c in drop_miss_cols if c in X_test.columns], errors="ignore")

# --- (5) 학습 컬럼에 맞게 정렬 (가장 중요) ---
train_feature_cols = X.columns.tolist()
X_test = X_test.reindex(columns=train_feature_cols)

# --- (6) categorical 처리: 학습 때 기준으로 string 포함해서 잡기 ---
cat_features = X_train.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
if len(cat_features) > 0:
    X_test[cat_features] = X_test[cat_features].astype("string").fillna("__MISSING__")

print("X_test aligned:", X_test.shape)
print("Total NaN:", int(X_test.isna().sum().sum()))


In [ ]:
# ✅ 이미 위 셀에서 FINAL_THR=0.47로 고정했다고 했으니 그대로 사용
print("FINAL_THR used for submission:", FINAL_THR)

proba_test = model_v2.predict_proba(X_test)[:, 1]
pred_test  = (proba_test >= FINAL_THR).astype(int)

print("pred_test 1비율:", float(pred_test.mean()))
print(pd.Series(pred_test).value_counts())

# ✅ ID 기준 안전 merge
temp = pd.DataFrame({"ID": test["ID"].values, "completed": pred_test})
sub_out = sub.drop(columns=["completed"], errors="ignore").merge(temp, on="ID", how="left")

# 점검
nan_cnt = int(sub_out["completed"].isna().sum())
dup_cnt = int(sub_out["ID"].duplicated().sum())
print("merge NaN(정상 0):", nan_cnt)
print("ID dup(정상 0):", dup_cnt)
print("label dist:", sub_out["completed"].value_counts().to_dict())

# 저장
out_path = f"submission_v2lite_thr{str(FINAL_THR).replace('.','p')}.csv"
sub_out.to_csv(out_path, index=False)
print("Saved:", out_path)

# (권장) flipped도 같이 저장해서 방향 확인
sub_flip = sub_out.copy()
sub_flip["completed"] = 1 - sub_flip["completed"]
flip_path = f"submission_v2lite_thr{str(FINAL_THR).replace('.','p')}_flipped.csv"
sub_flip.to_csv(flip_path, index=False)
print("Saved:", flip_path)
